# Free Video Transcriber (Google Colab Version)

This notebook allows you to use Google Colab's **free GPUs** to transcribe YouTube or Facebook videos incredibly fast without any limits.

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Run the first cell to install the required libraries.
3. Run the second cell, paste your URL, and get your transcript!

In [ ]:
# Install dependencies (Run this first!)
!pip install yt-dlp openai-whisper
!apt update && apt install ffmpeg -y

In [ ]:
# Run the transcriber
import yt_dlp
import whisper
import os

VIDEO_URL = "" # @param {type:"string"}
MODEL_SIZE = "large" # @param ["tiny", "base", "small", "medium", "large"]

if not VIDEO_URL:
    print("Please enter a valid VIDEO_URL above.")
else:
    print(f"Downloading audio from {VIDEO_URL}...")
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': '%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }],
        'quiet': True,
    }
    
    audio_file = None
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(VIDEO_URL, download=True)
            audio_file = f"{info['id']}.m4a"
            
        print(f"Loading Whisper model '{MODEL_SIZE}' (this uses GPU if available)...")
        model = whisper.load_model(MODEL_SIZE)
        
        print("Transcribing...")
        result = model.transcribe(audio_file)
        transcript = result["text"]
        
        with open("transcript.txt", "w", encoding="utf-8") as f:
            f.write(transcript)
            
        print("\n=== TRANSCRIPT ===\n")
        print(transcript)
        print("\n==================\n")
        print("Transcription saved to 'transcript.txt' in the Files tab on the left.")
        
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        if audio_file and os.path.exists(audio_file):
            os.remove(audio_file)
